# Prepare and monitor a model customization job responsibly

Validate synthetic chat data, construct a reproducible training manifest, estimate optimizer steps, and test a bounded job monitor against success, failure, and timeout histories before submitting GPU work.

**Audience:** Python developers building and evaluating enterprise AI applications. Familiarity with dictionaries, functions and JSON is assumed.

**Learning objectives:** split synthetic chat pairs without overlapping exported examples; validate and fingerprint the training wire format; estimate optimizer steps and reject failed, regressing, or timed-out job histories.

**Prerequisites:** Understand Python collections, JSON, and exception handling. The default preparation and monitor use only the standard library. Live training additionally requires deployment-supported NVIDIA tooling, an approved model and dataset, and GPU capacity; it is not executed here.

**Setup:** use patched Python 3.12 (3.12.16 or newer) and the audited environment in the [collection README](../../README.md). Run cells in order in a fresh kernel. All default data is newly invented for this tutorial. No customer records, private repositories, credentials, downloads, or paid calls are required. Optional integrations are disabled in their own cells and never enabled by an environment variable alone.

**Execution boundary:** the results below describe the local computation shown in this notebook. They do not establish hosted-model quality or production integration success.



Copyright IBM Corp. 2026

SPDX-License-Identifier: Apache-2.0

**License and AI-assistance notice.** Source code is licensed under Apache License 2.0. Modified for open-source preparation on 2026-10-23. This notebook may include contributions involving Anthropic Claude Opus 4+, Claude Sonnet 3.7+, and OpenAI GPT-4+ models; exact per-file model provenance is unverified. See [AI disclosure](../../AI_DISCLOSURE.md), [license](../../LICENSE), and [attribution](../../NOTICE.md). Human contributors remain responsible for review, rights and DCO certification.


## 1. Build deterministic synthetic training data

The task classifies an invented equipment reading into normal, inspect, or stop. Labels come from an explicit policy, not from a model. Devices are separated between splits, and each device has distinct measured temperatures. Assert disjointness of the actual exported messages: disjoint sidecar device IDs alone would not prevent duplicate training and validation examples.


In [ ]:
import hashlib
import json
import math

rows = []
for device in range(12):
    for base_temperature in (20, 40, 60, 80):
        # Disjoint measured values, not merely different sidecar device IDs.
        temperature = base_temperature + device / 10
        label = "normal" if temperature < 50 else "inspect" if temperature < 75 else "stop"
        rows.append(
            {
                "device_group": f"device-{device:02d}",
                "messages": [
                    {
                        "role": "user",
                        "content": f"Classify the fictional device reading: temperature={temperature} C.",
                    },
                    {"role": "assistant", "content": label},
                ],
            }
        )
train = [r for r in rows if int(r["device_group"].split("-")[-1]) < 9]
validation = [r for r in rows if r not in train]
assert len(train) == 36 and len(validation) == 12
assert {r["device_group"] for r in train}.isdisjoint(r["device_group"] for r in validation)
serialized_train = {json.dumps(r["messages"], sort_keys=True) for r in train}
serialized_validation = {json.dumps(r["messages"], sort_keys=True) for r in validation}
assert len(serialized_train) == 36 and len(serialized_validation) == 12
assert serialized_train.isdisjoint(serialized_validation)
print(
    {
        "training": len(train),
        "validation": len(validation),
        "labels": sorted({r["messages"][-1]["content"] for r in rows}),
    }
)

## 2. Validate the wire format and fingerprint the experiment

Validation rejects missing assistant messages, unknown roles, empty content, and oversized examples. Reject unknown per-message fields, including sidecar metadata, and export fresh objects containing only role and content. The sidecar manifest records group membership separately.


In [ ]:
def validate_example(row):
    if not isinstance(row, dict):
        raise ValueError("Expected an example object")
    messages = row.get("messages")
    if not isinstance(messages, list) or len(messages) != 2:
        raise ValueError("Expected user/assistant pair")
    if any(
        not isinstance(message, dict) or set(message) != {"role", "content"} for message in messages
    ):
        raise ValueError("Only role/content message fields are supported")
    if [m.get("role") for m in messages] != ["user", "assistant"]:
        raise ValueError("Invalid roles")
    if any(
        not isinstance(m.get("content"), str)
        or not m["content"].strip()
        or len(m["content"]) > 2048
        for m in messages
    ):
        raise ValueError("Invalid content")
    return {"messages": [{"role": m["role"], "content": m["content"]} for m in messages]}


def export(rows):
    return "".join(json.dumps(validate_example(r), sort_keys=True) + "\n" for r in rows)


training_bytes = export(train).encode()
validation_bytes = export(validation).encode()
manifest = {
    "schema": "messages-v1",
    "training_rows": len(train),
    "validation_rows": len(validation),
    "training_groups": sorted({row["device_group"] for row in train}),
    "validation_groups": sorted({row["device_group"] for row in validation}),
    "training_sha256": hashlib.sha256(training_bytes).hexdigest(),
    "validation_sha256": hashlib.sha256(validation_bytes).hexdigest(),
    "data_provenance": "authored synthetic thresholds",
}
try:
    validate_example({"messages": [{"role": "user", "content": "missing label"}]})
except ValueError:
    pass
else:
    raise AssertionError("Malformed row accepted")
# Metadata belongs outside the wire-format messages, and exports own their copies.
clean_pair = train[0]["messages"]
invalid_examples = [
    {"messages": [dict(clean_pair[0], private_note="invented sidecar marker"), clean_pair[1]]},
    {"messages": [clean_pair[0], dict(clean_pair[1], weight=-100)]},
    {"messages": ["not an object", clean_pair[1]]},
    {"messages": [dict(clean_pair[0], content=""), clean_pair[1]]},
    {"messages": [dict(clean_pair[0], content="x" * 2049), clean_pair[1]]},
    {"messages": [clean_pair[0], dict(clean_pair[1], role="system")]},
]
for invalid in invalid_examples:
    try:
        validate_example(invalid)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid message format was exported")
exported = validate_example(train[0])
assert set(exported) == {"messages"}
exported["messages"][0]["content"] = "Mutated exported copy"
assert train[0]["messages"][0]["content"] != "Mutated exported copy"
assert manifest["training_sha256"] != manifest["validation_sha256"]
assert set(manifest["training_groups"]).isdisjoint(manifest["validation_groups"])
print(json.dumps(manifest, indent=2))

## 3. Plan compute separately from provisioning

This calculation counts optimizer steps for a stated global batch size and number of epochs. It excludes padding, sequence packing, distributed synchronization, and evaluation. No GPU availability or dollar cost is inferred.


In [ ]:
plan = {
    "epochs": 3,
    "micro_batch": 2,
    "gradient_accumulation": 4,
    "workers": 1,
    "learning_rate": 0.0001,
}
global_batch = plan["micro_batch"] * plan["gradient_accumulation"] * plan["workers"]
steps_per_epoch = math.ceil(len(train) / global_batch)
optimizer_steps = steps_per_epoch * plan["epochs"]
assert global_batch == 8 and optimizer_steps == 15
job_spec = {
    "training": {
        "training_type": "sft",
        "finetuning_type": "lora",
        "lora": {"rank": 8, "alpha": 16},
    },
    "schedule": {"epochs": plan["epochs"]},
    "batch": {
        "global_batch_size": global_batch,
        "micro_batch_size": plan["micro_batch"],
    },
    "optimizer": {"learning_rate": plan["learning_rate"]},
}
print(
    {
        "global_batch": global_batch,
        "optimizer_steps_estimate": optimizer_steps,
        "spec": job_spec,
    }
)

## 4. Test a bounded job-status monitor

These are normalized tutorial states, not a claim about one provider’s response enum. A provider adapter must map its documented states to this contract. Polling observes snapshots: the first observation may already be terminal, and a queued observation may be followed directly by succeeded. Accept reachable forward states and reject regressions such as running to queued. Repeated pending states consume the polling budget; failures and illegal states raise.


In [ ]:
TRANSITIONS = {
    None: {"queued", "running", "succeeded", "failed", "cancelled"},
    "queued": {"queued", "running", "succeeded", "failed", "cancelled"},
    "running": {"running", "succeeded", "failed", "cancelled"},
}


def monitor(read_status, max_polls=6):
    if type(max_polls) is not int or max_polls < 1:
        raise ValueError("Positive integer polling budget required")
    previous = None
    history = []
    for attempt in range(max_polls):
        state = read_status()
        if state not in TRANSITIONS.get(previous, set()):
            raise ValueError("Illegal job transition")
        history.append(state)
        if state == "succeeded":
            return {"status": state, "polls": attempt + 1, "history": history}
        if state in {"failed", "cancelled"}:
            raise RuntimeError("Job did not complete successfully")
        previous = state
    raise TimeoutError("Polling budget exhausted; job may still be active")


def reader(sequence):
    iterator = iter(sequence)
    last = None

    def read():
        nonlocal last
        last = next(iterator, last)
        return last

    return read


completed = monitor(reader(["queued", "running", "running", "succeeded"]))
for sequence, error in [
    (["queued", "failed"], RuntimeError),
    (["queued"] * 9, TimeoutError),
    (["running", "queued"], ValueError),
    (["unknown"], ValueError),
]:
    try:
        monitor(reader(sequence))
    except error:
        pass
    else:
        raise AssertionError("Failure path was not enforced")
for sequence in [["queued", "succeeded"], ["running", "succeeded"], ["succeeded"]]:
    assert monitor(reader(sequence))["status"] == "succeeded"
assert completed["polls"] == 4
print(completed)

## 5. Prepare an explicit handoff for a provisioned customization platform

The current NVIDIA documentation describes `NeMoHelix` and `AutomodelJobInput`, but on 2026-10-01 the public `nemo-helix` package is a placeholder and `nemo-automodel-plugin` is not available on PyPI. This collection therefore does not advertise a working pip-installable live adapter. Use the NVIDIA deployment-provided tooling and the linked current training guide in a separately approved environment.

The local handoff below contains reproducible data hashes and a documented job-spec shape. Replace model and FileSet references with resources you created and validated on that platform, validate against its live schema, and review authentication, GPU quota, model licensing and costs before submission. No executable live submission branch is included until a supported, auditable SDK or API profile is available.


In [ ]:
handoff = {
    "workspace": "tutorial-workspace",
    "model": "tutorial-workspace/approved-model",
    "dataset": {
        "training": "tutorial-workspace/synthetic-training",
        "validation": "tutorial-workspace/synthetic-validation",
    },
    "spec": job_spec,
    "provenance": manifest,
}
handoff_json = json.dumps(handoff, sort_keys=True, indent=2)
assert json.loads(handoff_json)["provenance"]["training_rows"] == 36
print(
    {
        "handoff_sha256": hashlib.sha256(handoff_json.encode()).hexdigest(),
        "live_submission": "requires deployment-supported tooling; not run",
    }
)

## Expected local results

The dataset has 36 training and 12 validation rows. A global batch of eight over three epochs yields an estimate of 15 optimizer steps. The successful fixture completes after four polls; failure, timeout and illegal-transition fixtures are rejected. No model is trained.

**Try next:** change batch size and dataset grouping, then explain how drop-last behavior would alter the step estimate.


## Interpretation, limitations, and cleanup

The monitor proves application control flow, not hosted training success. The label rule is deliberately easy; fine-tuning this task is educational rather than necessary. Provider states, authentication, quotas, optimizer semantics, and model licensing need deployment-specific review. Keep a held-out test set beyond the validation split for a real project.

The default run keeps its artifacts in memory. Restart the kernel to release them. If you opt into a remote integration, review and remove its synthetic test records using that service's documented controls; this notebook does not delete remote data. Never save credential values or private provider output into notebook cells.


## References and provenance

All scenario text, values, evaluation cases and tutorial code were authored for this public example. No external dataset is bundled. Product names identify optional integrations and do not imply endorsement. API references were checked on 2026-10-01.

- [NeMo Helix model/training resources](https://docs.nvidia.com/nemo-helix/documentation/customizer-reference/tutorials/understanding-models-and-training/)
- [NeMo training configuration](https://docs.nvidia.com/nemo-helix/v0.3.0/documentation/customizer-reference/manage-customization-jobs/training-configuration/)
